In [0]:
from pyspark.sql.functions import col,when,sum as s, avg as a, count as c
from pyspark.sql.window import Window
from pyspark.sql import *
from datetime import datetime
from zoneinfo import ZoneInfo

In [0]:
storage = "abfss://onlinestore@onlinestorage2026.dfs.core.windows.net/"

product_sliver_versions = [f.name for f in dbutils.fs.ls(storage + "sliver/product_sales/") if f.isDir()]
monthly_sliver_versions = [f.name for f in dbutils.fs.ls(storage + "sliver/monthly_sales/") if f.isDir()]

product_latest_version = sorted(product_sliver_versions)[-1]
monthly_latest_version = sorted(monthly_sliver_versions)[-1]

sliver_path_product = storage + "sliver/product_sales/" + product_latest_version
sliver_path_monthly = storage + "sliver/monthly_sales/" + monthly_latest_version

In [0]:
product_df = spark.read.format("delta").load(sliver_path_product)
display(product_df)

In [0]:
monthly_df = spark.read.format("delta").load(sliver_path_monthly)
display(monthly_df)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS onlineretail.onlineretail_schema.productS_Sales(
  event_id STRING,
  event_timestamp STRING,
  product_id STRING,
  product_type STRING,
  month STRING,
  year INT,
  net_quantity INT,
  gross_sales DOUBLE,
  discounts DOUBLE,
  returns DOUBLE,
  total_net_sales DOUBLE
)

In [0]:
product_df.write.mode("overwrite").saveAsTable("onlineretail.onlineretail_schema.productS_Sales")

In [0]:
%sql
select * from onlineretail.onlineretail_schema.productS_Sales

In [0]:
%sql
CREATE TABLE IF NOT EXISTS onlineretail.onlineretail_schema.monthlySales(
  event_id STRING,
  event_timestamp STRING,
  month STRING,
  year INT,
  total_orders INT,
  gross_sales DOUBLE,
  discounts DOUBLE,
  returns DOUBLE,
  net_sales DOUBLE,
  shipping DOUBLE,
  total_sales DOUBLE
)

In [0]:
monthly_df.write.mode("overwrite").saveAsTable("onlineretail.onlineretail_schema.monthlySales")

In [0]:
%sql
select * from onlineretail.onlineretail_schema.monthlySales